In [ ]:
# This example demonstrates how to use the SQLiteInterface and SQLiteMigrator
# classes to manage a user database with migrations.

import logging
import os
from pathlib import Path

from sqlite_manager.interface import SQLiteInterface
from sqlite_manager.migrator import SQLiteMigrator
from user_manager import UserManager

%load_ext autoreload
%autoreload 2

PROJECT_DIR = Path(os.getcwd()).resolve()
DB_PATH = PROJECT_DIR / "users.sqlite"
MIGRATIONS_DIR = PROJECT_DIR / "migrations"
BACKUP_DIR = PROJECT_DIR / "backups"

logging.basicConfig(level=logging.INFO)


def create_user_database() -> SQLiteInterface:
    """Creates the user database and applies migrations.

    This will create the SQLite database file if it doesn't exist, and apply any
    pending migrations to ensure the database schema is up to date. It will also
    create a backup of the database before applying migrations.
    """

    sql_db = SQLiteInterface(DB_PATH)
    migrator = SQLiteMigrator(DB_PATH, MIGRATIONS_DIR, BACKUP_DIR)
    migrator.migrate()

    return sql_db

In [2]:
# Create the user database and apply migrations
user_db = create_user_database()

# Initialize the UserManager with the SQLiteInterface
user_manager = UserManager(user_db)

INFO:sqlite_manager.migrator:Backed up database to /home/jverster/Projects/Other/sqlite-manager/examples/user_mamagement/backups/backup_v0_2025-04-07T073649.sqlite3...
INFO:sqlite_manager.migrator:Successfully migrated database to version 1


In [3]:
# Create an admin and a regular user
user_manager.create_user(username="admin", password="Pass1234!")
user_manager.create_user(username="regular_user", password="UserPass1234!")

INFO:sqlite_manager.crud:Inserted new record into users
INFO:sqlite_manager.crud:Inserted new record into users


True

In [4]:
# Authenticate users
admin = user_manager.authenticate("admin", "Pass1234!")
user = user_manager.authenticate("regular_user", "UserPass1234!")

INFO:user_manager:User authenticated: admin
INFO:user_manager:User authenticated: regular_user


In [5]:
# Read user data using a filter
user_manager.read(
    filter={"username": "regular_user"}  # or {"user_id": 1} for example
)

{'user_id': 2,
 'username': 'regular_user',
 'role': 'user',
 'hashed_password': b'$2b$12$J9/E.UVjg.i0lpvpIgsnKeUlDzqNsqAtt2xYfQW8vFyHbmIjYTC0W',
 'last_login': '2025-04-07 07:36:50',
 'created_at': '2025-04-07 07:36:49',
 'activated': 1}

In [6]:
# List all users
user_manager.list_users()

[{'user_id': 1,
  'username': 'admin',
  'role': 'user',
  'hashed_password': b'$2b$12$jN8t8tcUFHnyDEAHvgV17e/8XMn9C/sAhvy1.BIPxH0408IXWJjm6',
  'last_login': '2025-04-07 07:36:50',
  'created_at': '2025-04-07 07:36:49',
  'activated': 1},
 {'user_id': 2,
  'username': 'regular_user',
  'role': 'user',
  'hashed_password': b'$2b$12$J9/E.UVjg.i0lpvpIgsnKeUlDzqNsqAtt2xYfQW8vFyHbmIjYTC0W',
  'last_login': '2025-04-07 07:36:50',
  'created_at': '2025-04-07 07:36:49',
  'activated': 1}]

In [7]:
user_manager.update_user(
    user_id=2,
    username="regular_user_updated",  # Change the username for the user with ID 2
    password="UpdatedPass1234!",  # Update the password for the user
    activated=False,  # Deactivate the user
)

# Read the user data again to see the changes
user_manager.read(filter={"username": "regular_user_updated"})

INFO:sqlite_manager.crud:Updated record in users: ID=2


{'user_id': 2,
 'username': 'regular_user_updated',
 'role': 'user',
 'hashed_password': b'$2b$12$koGFV1Zo2d968RgoGkRtnePPXFgPnoOjUb8hNSzo8q33aAoD39rA.',
 'last_login': '2025-04-07 07:36:50',
 'created_at': '2025-04-07 07:36:49',
 'activated': 0}